## A/B testing

This a small A/B testing project based on different company's data for the demo purposes. Since the ecommerce dataset did not have the data like this, I decided to use the data from [here](https://www.kaggle.com/datasets/farhadzeynalli/online-advertising-effectiveness-study-ab-testing).

Before analyzing the experiment results, the hypothesis is as below:

*Advertisement on company website will have no increase in sales.*

This is our null hypothesis (H0), which assumes that there will be no significant difference between the control group (PSA) and treatment (Ads).

After running the A/B test, we can either:
reject H0 — there is a significant difference between control and treatment.
Or fail to reject H0 — we couldn’t detect a significant difference between control and treatment.

The dataset includes the following columns:

- TEST_GROUP
- MADE_PURCHASE       ← primary conversion metric
- DAYS_WITH_MOST_AD   ← behavioral metric
- PEAK_AD_HOURS       ← behavioral metric
- AD_COUNT            ← exposure metric

In [ ]:
# Import Python packages
from snowflake.snowpark.context import get_active_session
import pandas as pd
import numpy as np

# Get the current credentials
session = get_active_session()
ab = session.table("PRODUCT_ANALYTICS.UNRELATED_AB_TESTING_DATA.AB_TEST_DATA").to_pandas()

In [ ]:
print(ab.groupby("TEST_GROUP").agg(
    users=("CUSTOMERID", "count"),
    purchase_rate=("MADE_PURCHASE", "mean"),
    avg_ad_count=("AD_COUNT", "mean"),
    avg_days=("DAYS_WITH_MOST_AD", "mean"),
    avg_peak_hours=("PEAK_AD_HOURS", "mean")
))

In [ ]:
summary = (
    ab.groupby("TEST_GROUP")
      .agg(
          users=("CUSTOMERID", "count"),
          purchasers=("MADE_PURCHASE", "sum")
      )
)

summary["conversion"] = (
    summary["purchasers"] / summary["users"]
)

In [ ]:
print(summary.head())

Conversion rates are roughly:

Control: ~3.2% <br>
Treatment: ~6.66%

So the treatment appears to have more than doubled the conversion rate. Let's check the statistical significance:

In [ ]:
from statsmodels.stats.proportion import proportions_ztest, proportion_confint

control = summary.loc["psa"]
treatment = summary.loc["ad"]

count = [
    control["purchasers"],
    treatment["purchasers"]
]

nobs = [
    control["users"],
    treatment["users"]
]

# z-test
z_stat, pval = proportions_ztest(count, nobs)
(lower_con, lower_treat), (upper_con, upper_treat) = proportion_confint(count, nobs=nobs, alpha=0.05)

print(f'z statistic: {z_stat:.2f}')
print(f'p-value: {pval:.28f}')
print(f'ci 95% for control group: [{lower_con:.3f}, {upper_con:.3f}]')
print(f'ci 95% for treatment group: [{lower_treat:.3f}, {upper_treat:.3f}]')

In [ ]:
control_rate = control["conversion"]
treatment_rate = treatment["conversion"]

absolute_lift = treatment_rate - control_rate

relative_lift = (
    treatment_rate / control_rate - 1
)
print(f"Absolute rift is {round(absolute_lift*100, 2)}%.")
print(f"Relative rift is {round(relative_lift*100, 2)}%.")

Based on the results, we reject the null hypothesis at conventional significance levels (α = 0.05).

The **treatment increased conversion by 3.43%**, corresponding to a ~106% relative increase compared with the control group. The difference was statistically significant (z = −10.59, p < 0.001), with non-overlapping 95% confidence intervals.

Let's now investigate heterogeneous treatment effects by asking the question:

### Does the treatment work equally well across different levels of ad exposure?

In [ ]:
ab.groupby(
    ["TEST_GROUP", "AD_COUNT"]
)["MADE_PURCHASE"].mean()

In [ ]:
ab["AD_EXPOSURE_BUCKET"] = pd.cut(
    ab["AD_COUNT"],
    bins=[-1, 2, 5, 10],
    labels=["Low", "Medium", "High"]
)

In [ ]:
ab["AD_EXPOSURE_BUCKET"].value_counts()

In [ ]:
ab.groupby(["AD_EXPOSURE_BUCKET", "TEST_GROUP"])["MADE_PURCHASE"].mean()

In [ ]:
purchase_rates = (
    ab.groupby(["AD_EXPOSURE_BUCKET", "TEST_GROUP"])["MADE_PURCHASE"]
      .mean()
      .unstack()
)

print(purchase_rates)

In [ ]:
lift = pd.DataFrame({
    "control_rate": purchase_rates["psa"],
    "treatment_rate": purchase_rates["ad"],
})

lift["absolute_lift"] = (
    lift["treatment_rate"] - lift["control_rate"]
)

lift["relative_lift"] = (
    lift["treatment_rate"] - lift["control_rate"]
) / lift["control_rate"]

print(lift)

Based on absolute rifts, treatment effect looks quite consistent across exposure levels, which would support the non-significant interaction we see. Also, these descriptive lifts don't prove that the effects differ statistically. Let's do the statistical treatment × exposure interaction test.

### Formal interaction test for <code>AD_COUNT</code>

In [ ]:
ab["MADE_PURCHASE"] = ab["MADE_PURCHASE"].astype(int)
ab["TEST_GROUP"] = pd.Categorical(
    ab["TEST_GROUP"],
    categories=["psa", "ad"]
)

In [ ]:
import statsmodels.formula.api as smf

model = smf.logit(
    "MADE_PURCHASE ~ C(TEST_GROUP) * AD_COUNT",
    data=ab
).fit()

print(model.summary())

Since <code>C(TEST_GROUP)[T.ad]:AD_COUNT</code> shows how much the treatment effect changes as ad exposure increases, the results show that the p-value is 0.705, which is very far above 0.05. We can conclude that **there is no statistically significant evidence that treatment effectiveness varies with ad exposure**. 

The treatment substantially increases purchase probability, but the magnitude of this treatment effect does not appear to depend on the number of ads a user sees.

### Does the treatment effect vary depending on how many days users were exposed to ads?

We want to see if the treatment advantage appears to change as exposure days increase.

In [ ]:
ab["DAYS_WITH_MOST_AD"].describe()

In [ ]:
days_rates = (
    ab.groupby(["DAYS_WITH_MOST_AD", "TEST_GROUP"])["MADE_PURCHASE"]
      .mean()
      .unstack()
)

print(days_rates)

In [ ]:
days_lift = pd.DataFrame({
    "control_rate": days_rates["psa"],
    "treatment_rate": days_rates["ad"]
})

days_lift["absolute_lift"] = (
    days_lift["treatment_rate"] -
    days_lift["control_rate"]
)

days_lift["relative_lift"] = (
    days_lift["absolute_lift"] /
    days_lift["control_rate"]
)

print(days_lift)

In [ ]:
import matplotlib.pyplot as plt

days_rates.plot(kind="line", marker="o")

plt.xlabel("Number of days exposed to ads")
plt.ylabel("Purchase rate")
plt.title("Purchase Rate by Ad Exposure Days and Treatment Group")
plt.legend(title="Test Group")
plt.show()

The treatment advantage doesn't seem to stay constant or increase with passing days. Let's verify through a test:

### Formal interaction test for <code>DAYS_WITH_MOST_AD</code>

In [ ]:
model_days = smf.logit(
    "MADE_PURCHASE ~ C(TEST_GROUP) * DAYS_WITH_MOST_AD",
    data=ab
).fit()

print(model_days.summary())

With the p-value for <code>C(TEST_GROUP)[T.ad]:DAYS_WITH_MOST_AD</code> = 0.387, which is more than 0.05, there is no statistically significant evidence that treatment effectiveness changes with exposure days.